# Exercise 1
## Designing CNNs for Embedded Deployment

---

### Objective

In this exercise, we will learn that:

> A neural network that works on a PC is NOT automatically suitable for a microcontroller (MCU).

We will:
- Train a CNN on a gesture dataset
- Analyze its behavior
- Optimize its architecture
- Understand tradeoffs between accuracy and efficiency

---

### Embedded Constraints

Even though the MCXN has
- Flash: 2MB
- RAM: 640 KB

Memory available for ML would usually be more like:
- Flash: 200 KB – 800 KB
- RAM: 50 KB – 200 KB

So, to make things interesting, we will assume we have:
- 256 KB for the model itself
- 64 KB for inference


## Setup

- Install dependencies
- Import libraries
- Check that CUDA is working

In [1]:
!pip install torch torchvision torchinfo pandas matplotlib grad-cam

import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from torchinfo import summary
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
import numpy as np


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## Dataset: Sign Language MNIST

- Each image represents a hand gesture corresponding to a letter.

In [1]:
!wget -O sign_mnist_train.csv https://huggingface.co/spaces/Kolpitor/AI_ML/resolve/194ee29a6b30b8224bcb8d78889ea8a4d8e1eaf4/sign_mnist_train.csv
!wget -O sign_mnist_test.csv  https://huggingface.co/spaces/Kolpitor/AI_ML/resolve/194ee29a6b30b8224bcb8d78889ea8a4d8e1eaf4/sign_mnist_test.csv

In [ ]:
# Limit only to specific classes/letters
SELECTED_CLASSES = [0, 1, 4, 11, 24]
LABEL_NAMES = ["A", "B", "E", "L", "Y"]

NUM_CLASSES = len(SELECTED_CLASSES)

class SignDataset(Dataset):
    def __init__(self, file):
        data = pd.read_csv(file)

        labels = data.iloc[:, 0].values
        images = data.iloc[:, 1:].values.reshape(-1, 28, 28)

        # filter
        mask = [l in SELECTED_CLASSES for l in labels]
        images = images[mask]
        labels = labels[mask]

        # remap labels
        self.label_map = {c: i for i, c in enumerate(SELECTED_CLASSES)}
        self.images = images
        self.labels = [self.label_map[l] for l in labels]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = torch.tensor(self.images[idx], dtype=torch.float32).unsqueeze(0) / 255.0
        img = F.interpolate(img.unsqueeze(0), size=(32,32)).squeeze(0)
        label = torch.tensor(self.labels[idx])
        return img, label

train_dataset = SignDataset("sign_mnist_train.csv")
test_dataset  = SignDataset("sign_mnist_test.csv")

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=64)

## Visualize the dataset

- What do you look at when deciding what you see?
- What would the neural network look at?
- What classes would be the most difficult to recognize?


In [ ]:
def show_class_samples(dataset, selected_classes, label_names):
    plt.figure(figsize=(10, len(selected_classes)*2))

    for row, cls in enumerate(selected_classes):
        shown = 0

        for i in range(len(dataset)):
            img, label = dataset[i]

            # label is already remapped (0..N-1)
            if label.item() == row:
                plt.subplot(len(selected_classes), 4, row*4 + shown + 1)
                plt.imshow(img.squeeze(), cmap='gray')
                plt.title(label_names[row])
                plt.axis('off')

                shown += 1
                if shown == 4:
                    break

    plt.tight_layout()
    plt.show()


show_class_samples(train_dataset, SELECTED_CLASSES, LABEL_NAMES)


## Helper functions

In [ ]:
def count_params(model):
    return sum(p.numel() for p in model.parameters())

def model_flash_size_bytes(model):
    return count_params(model) * 4 # FP32 => 4 bytes

def train(model, epochs=10):
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    loss_fn = nn.CrossEntropyLoss()

    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()
            loss = loss_fn(model(x), y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
        print(f"Epoch {epoch+1}: {total_loss:.2f}")

def evaluate(model):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            pred = model(x).argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)

    return correct / total

## Baseline CNN

In [ ]:
class BaselineCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.features = nn.Sequential(
            # (Input, Output, Kernel, Padding)
            nn.Conv2d(1, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 32, 3, padding=1),
            nn.ReLU()
        )

        self.classifier = nn.Sequential(
            # Create a one long vector from 32 feature maps of size 8*8
            nn.Flatten(),
            # Condense into 64 values
            nn.Linear(32 * 8 * 8, 64),
            nn.ReLU(),
            # Condense into [num_classes] of values -> getting logits
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

## Train the model

In [ ]:
# Start from an untrained model every training
baseline = BaselineCNN(NUM_CLASSES).to(device)

train(baseline, epochs=10)
baseline_acc = evaluate(baseline)

print("Accuracy:", baseline_acc)
print("Parameters:", count_params(baseline))
print("Size (bytes):", model_flash_size_bytes(baseline))

## Visualize predictions

In [ ]:
def visualize_predictions_with_confidence(model, loader, label_names):
    model.eval()

    images, labels = next(iter(loader))
    images, labels = images.to(device), labels.to(device)

    with torch.no_grad():
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        preds = probs.argmax(dim=1)

    images = images.cpu()
    preds = preds.cpu()
    labels = labels.cpu()
    probs = probs.cpu()

    plt.figure(figsize=(12, 4))

    for i in range(8):
        plt.subplot(2, 4, i+1)
        plt.imshow(images[i].squeeze(), cmap='gray')

        pred_label = label_names[preds[i]]
        true_label = label_names[labels[i]]
        confidence = probs[i][preds[i]].item()

        plt.title(f"P:{pred_label} ({confidence:.2f})\nT:{true_label}")
        plt.axis('off')

    plt.tight_layout()
    plt.show()


visualize_predictions_with_confidence(baseline, test_loader, LABEL_NAMES)

## Explainability - Grad-CAM

- Let's see what the model actually looked at when deciding the class

In [ ]:
import random

def show_random_gradcam(model, dataset, label_names, feature_layer):
    # feature_layer is the most important parameter here, it tells GradCAM what layer we inspect

    model.eval()

    idx = random.randint(0, len(dataset)-1)

    image, label = dataset[idx]

    input_tensor = image.unsqueeze(0).to(device)

    target_layers = [model.features[feature_layer]]

    cam = GradCAM(
        model=model,
        target_layers=target_layers
    )

    pred = model(input_tensor).argmax(dim=1).item()

    grayscale_cam = cam(
        input_tensor=input_tensor,
        targets=[ClassifierOutputTarget(pred)]
    )[0]

    image_np = image.squeeze().numpy()

    plt.figure(figsize=(8,4))

    plt.subplot(1,2,1)
    plt.imshow(image_np, cmap='gray')
    plt.title(
        f"True: {label_names[label]}"
    )
    plt.axis('off')

    plt.subplot(1,2,2)
    plt.imshow(image_np, cmap='gray')
    plt.imshow(
        grayscale_cam,
        cmap='jet',
        alpha=0.5
    )

    plt.title(
        f"Pred: {label_names[pred]}"
    )
    plt.axis('off')

    plt.tight_layout()
    plt.show()

    print(f"Dataset index: {idx}")

In [ ]:
show_random_gradcam(baseline,test_dataset,LABEL_NAMES,6)

## Challenge: Fit into MCU Constraints

Assume:
- Flash ≤ 256 KB

### Tasks:

1. Does the model fit?
2. Where are most parameters?
3. Modify the model to reduce size!

## Better for Embedded: Depthwise Separable Conv

Instead of:
- heavy convolution

We use:
- Depthwise (per channel)
- Pointwise (1x1)

"Each feature map gets its own private 3×3 convolution."

In [ ]:
class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.depthwise = nn.Conv2d(in_ch, in_ch, 3, padding=1, groups=in_ch)
        self.pointwise = nn.Conv2d(in_ch, out_ch, 1)

    def forward(self, x):
        return self.pointwise(self.depthwise(x))

## Optimized Model

In [ ]:
class OptimizedCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.features = nn.Sequential(
            DepthwiseSeparableConv(1, 8),
            nn.ReLU(),
            nn.MaxPool2d(2),

            DepthwiseSeparableConv(8, 16),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 8 * 8, 16),
            nn.ReLU(),
            nn.Linear(16, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))



## Train optimized model

In [ ]:
# Start from an untrained model every training
optimized = OptimizedCNN(NUM_CLASSES).to(device)

train(optimized, epochs=10)
optimized_acc = evaluate(optimized)

print("Optimized accuracy:", optimized_acc)
print("Parameters:", count_params(optimized))
print("Size (bytes):", model_flash_size_bytes(optimized))

## Comparison

In [ ]:
print("Baseline params:", count_params(baseline))
print("Optimized params:", count_params(optimized))

print("Baseline acc:", baseline_acc)
print("Optimized acc:", optimized_acc)


In [ ]:
show_random_gradcam(optimized,test_dataset,LABEL_NAMES,3)

## Model inspection

### Baseline model

In [ ]:
print(summary(baseline, input_size=(1,1,32,32)))

### Optimized model

In [ ]:
print(summary(optimized, input_size=(1,1,32,32)))

## Next Exercise

We will:
- convert the model to ONNX
- apply quantization
- convert to TFLite


## Preparation for Excercise 2 - model export

A PyTorch model is Python code, not a stand-alone file. We will need to export only the weights of the model and define the architecture again.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
MODEL_PATH = "/content/drive/MyDrive/embedded_ai/optimized_model.pth"

!mkdir -p /content/drive/MyDrive/embedded_ai/

torch.save(
    optimized.state_dict(),
    MODEL_PATH
)

print("Saved:", MODEL_PATH)
